In [ ]:
from __future__ import annotations


class AssessmentService:

    def __init__(self, database):

        self.database = database

    def evaluate_open_ended(
        self,
        project_id: str,
        user_id: str,
        question: str,
        answer: str,
        expected_concepts: list[str],
        source_chunk_ids: list[str] | None = None,
    ):

        from app.services.ai_service import (
            AIService,
        )

        from app.services.retrieval_service import (
            RetrievalService,
        )

        from app.ai.evaluator import (
            AnswerEvaluator,
        )

        retrieval = RetrievalService(
            database=self.database
        )

        chunks = []

        if source_chunk_ids:
            collection = self.database.collection(
                "document_chunks"
            )

            documents = collection.find(
                {
                    "chunk_id": {
                        "$in": source_chunk_ids
                    },
                    "project_id": project_id,
                    "user_id": user_id,
                }
            )

            chunks = [
                retrieval._to_result(document)
                for document in documents
            ]

        if not chunks:

            chunks = retrieval.get_project_chunks(
                project_id=project_id,
                user_id=user_id,
                limit=10,
            )

        ai = AIService(
            database=self.database
        )

        evaluator = AnswerEvaluator(
            ai_service=ai
        )

        return evaluator.evaluate(
            question=question,
            answer=answer,
            expected_concepts=expected_concepts,
            evidence_chunks=chunks,
            user_id=user_id,
            project_id=project_id,
        )
